### PTV1 -> Basic Vector Attention + kNN

In [19]:
import os
import glob
import copy
import numpy as np
import laspy
import open3d as o3d
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_cluster import knn
from torch_scatter import scatter_softmax, scatter_add
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [20]:
# ----------------------------- Config -----------------------------
CFG = dict(
    train_dir        = "data/train",# folder with labeled .las files (classification: 0=other, 1=target)
    test_dir        =  "data/test",
    num_points      = 16384,        # points per training block (sphere crop)
    blocks_per_file = 4,           # random blocks sampled from each file per epoch
    batch_size      = 4,
    k_neighbors     = 16,
    epochs          = 10,
    lr              = 1e-3,
    weight_decay    = 1e-4,
    k_folds         = 5,
    num_classes     = 2,
    ckpt_dir        = "checkpoints",
)
os.makedirs(CFG["ckpt_dir"], exist_ok=True)

## Dataset

- Each `.las` is loaded **once** and cached: xyz (centered + scaled per file), labels, normals, height-above-floor.
- A training sample = one **sphere crop**: pick a random seed point, take its `num_points` nearest neighbors. This keeps local geometry intact for kNN attention.
- Features per point (7 channels): normalized xyz (3) + height above file z-min (1) + normal vector (3).
- Augmentation: random rotation around Z, random scale, jitter.

In [21]:
class CustomDataset(Dataset):
    def __init__(self, filepaths, num_points=8192, blocks_per_file=4, augment=True):
        super().__init__()
        self.filepaths = list(filepaths)
        self.num_points = num_points
        self.blocks_per_file = blocks_per_file
        self.augment = augment
        self._cache = {}

    def __len__(self):
        return len(self.filepaths) * self.blocks_per_file

    # ---------- loading & caching ----------
    def _load_file(self, filepath):
        if filepath in self._cache:
            return self._cache[filepath]

        las = laspy.read(filepath)
        xyz = np.column_stack((las.x, las.y, las.z)).astype(np.float64)
        labels = np.asarray(las.classification).astype(np.int64)

        # normals on the ORIGINAL geometry (before normalization)
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(xyz)
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.15, max_nn=30))
        pcd.orient_normals_to_align_with_direction([0.0, 0.0, 1.0])
        normals = np.asarray(pcd.normals).astype(np.float32)

        # height above the file's floor level (strong pile/floor/wall cue)
        height = (xyz[:, 2] - xyz[:, 2].min()).astype(np.float32)
        height = height / max(height.max(), 1e-6)          # 0..1 per file

        # per-file normalization: center + unit max-distance
        xyz = xyz - xyz.mean(axis=0, keepdims=True)
        scale = np.linalg.norm(xyz, axis=1).max()
        xyz = (xyz / max(scale, 1e-6)).astype(np.float32)

        entry = dict(xyz=xyz, labels=labels, normals=normals, height=height)
        self._cache[filepath] = entry
        return entry

    # ---------- augmentation ----------
    @staticmethod
    def _augment(xyz, normals):
        theta = np.random.uniform(0, 2 * np.pi)
        c, s = np.cos(theta), np.sin(theta)
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
        xyz = xyz @ R.T
        normals = normals @ R.T
        xyz = xyz * np.random.uniform(0.9, 1.1)                       # scale
        xyz = xyz + np.random.normal(0, 0.002, xyz.shape).astype(np.float32)  # jitter
        return xyz, normals

    def __getitem__(self, index):
        filepath = self.filepaths[index // self.blocks_per_file]
        d = self._load_file(filepath)
        n = len(d["xyz"])

        # sphere crop around a random seed point
        seed = np.random.randint(n)
        diff = d["xyz"] - d["xyz"][seed]
        dist2 = np.einsum("ij,ij->i", diff, diff)
        if n >= self.num_points:
            idx = np.argpartition(dist2, self.num_points - 1)[: self.num_points]
        else:
            idx = np.random.choice(n, self.num_points, replace=True)

        xyz     = d["xyz"][idx].copy()
        normals = d["normals"][idx].copy()
        height  = d["height"][idx].copy()
        labels  = d["labels"][idx].copy()

        if self.augment:
            xyz, normals = self._augment(xyz, normals)

        pos  = torch.from_numpy(xyz)                                   # (N, 3)
        feat = torch.from_numpy(
            np.column_stack((xyz, height[:, None], normals)))          # (N, 7)
        y    = torch.from_numpy(labels)                                # (N,)
        return pos, feat.float(), y

## Model — Point Transformer (vector attention, fixed)

The critical fix: `scatter_softmax(attn, center_index)` — softmax across each point's **k neighbors**, not across channels. Aggregation uses `scatter_add` grouped by center point.

In [22]:
class PointTransformerLayer(nn.Module):
    def __init__(self, channels, k=16):
        super().__init__()
        self.k = k
        self.linear_q = nn.Linear(channels, channels)
        self.linear_k = nn.Linear(channels, channels)
        self.linear_v = nn.Linear(channels, channels)
        self.pos_mlp = nn.Sequential(
            nn.Linear(3, channels), nn.ReLU(inplace=True), nn.Linear(channels, channels))
        self.attn_mlp = nn.Sequential(
            nn.Linear(channels, channels), nn.ReLU(inplace=True), nn.Linear(channels, channels))

    def forward(self, x, pos, batch):
        # knn(database, query, k, ...) -> edge[0]: query idx (center), edge[1]: database idx (neighbor)
        edge = knn(pos, pos, self.k, batch, batch)
        center, neighbor = edge[0], edge[1]

        q = self.linear_q(x)[center]                       # (E, C)
        k = self.linear_k(x)[neighbor]                     # (E, C)
        v = self.linear_v(x)[neighbor]                     # (E, C)

        pos_enc = self.pos_mlp(pos[center] - pos[neighbor])  # (E, C)

        attn = self.attn_mlp(q - k + pos_enc)              # (E, C)
        attn = scatter_softmax(attn, center, dim=0)        # <-- FIX: softmax over neighbors

        out = scatter_add(attn * (v + pos_enc), center, dim=0, dim_size=x.size(0))
        return out


class PTBlock(nn.Module):
    def __init__(self, channels, k=16):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.attn = PointTransformerLayer(channels, k)
        self.norm2 = nn.LayerNorm(channels)
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels * 2), nn.ReLU(inplace=True),
            nn.Linear(channels * 2, channels))

    def forward(self, x, pos, batch):
        x = x + self.attn(self.norm1(x), pos, batch)
        x = x + self.mlp(self.norm2(x))
        return x


class PointTransformerSeg(nn.Module):
    def __init__(self, in_channels=7, num_classes=2, k=16):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(in_channels, 64), nn.ReLU(inplace=True), nn.Linear(64, 64))
        self.block1 = PTBlock(64, k)
        self.up = nn.Linear(64, 128)
        self.block2 = PTBlock(128, k)
        self.block3 = PTBlock(128, k)
        self.head = nn.Sequential(
            nn.LayerNorm(128), nn.Linear(128, 64), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, pos, feat):
        # pos: (B, N, 3)   feat: (B, N, C)
        B, N, _ = pos.shape
        pos_flat = pos.reshape(-1, 3)
        x = feat.reshape(-1, feat.shape[-1])
        batch = torch.arange(B, device=pos.device).repeat_interleave(N)

        x = self.embed(x)
        x = self.block1(x, pos_flat, batch)
        x = F.relu(self.up(x))
        x = self.block2(x, pos_flat, batch)
        x = self.block3(x, pos_flat, batch)
        logits = self.head(x)
        return logits.reshape(B, N, -1)                    # (B, N, num_classes)

## Class weights & metrics

In [23]:
def compute_class_weights(filepaths, dataset, num_classes=2):
    counts = np.zeros(num_classes, dtype=np.int64)
    for fp in filepaths:
        labels = dataset._load_file(fp)["labels"]
        counts += np.bincount(labels, minlength=num_classes)
    freq = counts / counts.sum()
    weights = 1.0 / np.maximum(freq, 1e-6)
    weights = weights / weights.sum() * num_classes        # mean weight ~= 1
    print(f"Label counts: {counts.tolist()}  ->  class weights: {np.round(weights, 3).tolist()}")
    return torch.tensor(weights, dtype=torch.float32)


def iou_from_confusion(cm):
    inter = np.diag(cm).astype(np.float64)
    union = cm.sum(0) + cm.sum(1) - np.diag(cm)
    iou = inter / np.maximum(union, 1)
    return iou, iou.mean()

## Train / evaluate one fold

In [24]:
def run_epoch(model, loader, criterion, optimizer=None, num_classes=2):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, n_batches = 0.0, 0
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for pos, feat, labels in loader:
            pos, feat, labels = pos.to(device), feat.to(device), labels.to(device)
            logits = model(pos, feat)                              # (B, N, C)
            loss = criterion(logits.reshape(-1, num_classes), labels.reshape(-1))

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            n_batches += 1
            preds = logits.argmax(-1).reshape(-1).cpu().numpy()
            cm += confusion_matrix(labels.reshape(-1).cpu().numpy(), preds,
                                   labels=list(range(num_classes)))

    iou, miou = iou_from_confusion(cm)
    return total_loss / max(n_batches, 1), iou, miou, cm


def train_fold(fold, train_files, val_files, cfg):
    train_dataset = CustomDataset(train_files, cfg["num_points"], cfg["blocks_per_file"], augment=True)
    val_dataset   = CustomDataset(val_files,   cfg["num_points"], cfg["blocks_per_file"], augment=False)

    train_loader = DataLoader(train_dataset, batch_size=cfg["batch_size"], shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_dataset,   batch_size=cfg["batch_size"], shuffle=False, drop_last=False)

    weights = compute_class_weights(train_files, train_dataset, cfg["num_classes"]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    model = PointTransformerSeg(in_channels=7, num_classes=cfg["num_classes"],
                                k=cfg["k_neighbors"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                  weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])

    best_miou, best_state = -1.0, None
    for epoch in range(cfg["epochs"]):
        tr_loss, tr_iou, tr_miou, _ = run_epoch(model, train_loader, criterion, optimizer,
                                                cfg["num_classes"])
        va_loss, va_iou, va_miou, va_cm = run_epoch(model, val_loader, criterion, None,
                                                    cfg["num_classes"])
        scheduler.step()

        if va_miou > best_miou:
            best_miou = va_miou
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, os.path.join(cfg["ckpt_dir"], f"fold{fold}_best.pt"))

        print(f"Fold {fold} | Ep {epoch+1:3d}/{cfg['epochs']} | "
              f"tr loss {tr_loss:.4f} mIoU {tr_miou:.3f} | "
              f"va loss {va_loss:.4f} mIoU {va_miou:.3f} "
              f"(IoU other {va_iou[0]:.3f}, wood {va_iou[1]:.3f}) | best {best_miou:.3f}")

    model.load_state_dict(best_state)
    return model, best_miou

In [25]:
def spatial_sort_indices(xyz, cell=0.03):
    keys = np.floor(xyz / cell).astype(np.int64)
    return np.lexsort((keys[:, 2], keys[:, 1], keys[:, 0]))


@torch.no_grad()
def segment_full_cloud(filepath, model, cfg, out_path=None):
    model.eval()
    ds = CustomDataset([filepath], cfg["num_points"], 1, augment=False)
    d = ds._load_file(filepath)
    xyz, normals, height = d["xyz"], d["normals"], d["height"]
    n = len(xyz)

    order = spatial_sort_indices(xyz)
    preds = np.zeros(n, dtype=np.int64)

    npts = cfg["num_points"]
    for start in range(0, n, npts):
        idx = order[start: start + npts]
        pad = 0
        if len(idx) < npts:                                    # pad last chunk
            pad = npts - len(idx)
            idx = np.concatenate([idx, idx[np.random.randint(0, len(idx), pad)]])

        pos  = torch.from_numpy(xyz[idx]).unsqueeze(0).to(device)
        feat = torch.from_numpy(
            np.column_stack((xyz[idx], height[idx, None], normals[idx]))
        ).float().unsqueeze(0).to(device)

        p = model(pos, feat).argmax(-1).squeeze(0).cpu().numpy()
        if pad:
            idx, p = idx[:-pad], p[:-pad]
        preds[idx] = p

    if out_path is not None:
        las = laspy.read(filepath)
        las.classification = preds.astype(np.uint8)
        las.write(out_path)
        print(f"[Saved] {out_path}")

    cm = confusion_matrix(d["labels"], preds, labels=[0, 1])
    iou, miou = iou_from_confusion(cm)
    print(f"{os.path.basename(filepath)} | mIoU {miou:.3f} "
          f"(other {iou[0]:.3f}, wood {iou[1]:.3f})")
    print(classification_report(d["labels"], preds,
                                target_names=["other", "wood_powder"], digits=3))
    return preds

## Main — 5-fold cross-validation + held-out test

In [29]:
def visualize_test_predictions(test_files, model, cfg):
    for fp in test_files:
        preds = segment_full_cloud(fp, model, cfg)   # full-cloud inference

        d = CustomDataset([fp], cfg["num_points"], 1, augment=False)._load_file(fp)
        xyz = d["xyz"]   # already centered + normalized per file

        colors = np.zeros((len(xyz), 3))
        colors[preds == 1] = [0.0, 1.0, 0.0]   # Green = Wood Powder
        colors[preds == 0] = [1.0, 0.0, 0.0]   # Red   = Others

        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(xyz)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        frame=o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0,origin=(0,0,0))
        o3d.visualization.draw_geometries(
            [pcd,frame],
            window_name=f"{os.path.basename(fp)} | Green=Wood, Red=Others",
            width=1024, height=768)
        break

In [30]:
if __name__ == "__main__":
    model_path = "PointTransformerV1.pth"
    if os.path.exists(model_path):
        model = torch.load(model_path,weights_only=False)
        print("model loaded successfully")
        all_files = []
        for ext in ("*.las", "*.laz"):
            all_files.extend(glob.glob(os.path.join(CFG["test_dir"], ext)))
        visualize_test_predictions(all_files, model, CFG)
    else:
        all_files = []
        for ext in ("*.las", "*.laz"):
            all_files.extend(glob.glob(os.path.join(CFG["train_dir"], ext)))
        all_files = sorted(all_files)
        print(f"Found {len(all_files)} files")

        train_val_files, test_files = train_test_split(
            all_files, test_size=0.2, random_state=SEED)

        kf = KFold(n_splits=CFG["k_folds"], shuffle=True, random_state=SEED)
        fold_scores, best_model, best_score = [], None, -1.0

        for fold, (tr_idx, va_idx) in enumerate(kf.split(train_val_files)):
            tr_files = [train_val_files[i] for i in tr_idx]
            va_files = [train_val_files[i] for i in va_idx]
            print(f"\n===== Fold {fold+1}/{CFG['k_folds']} | "
                f"train {len(tr_files)} files, val {len(va_files)} files =====")

            model, miou = train_fold(fold, tr_files, va_files, CFG)
            fold_scores.append(miou)
            if miou > best_score:
                best_score, best_model = miou, model

        print(f"\nCV mIoU: {np.mean(fold_scores):.3f} +/- {np.std(fold_scores):.3f}  "
            f"per fold: {[round(s, 3) for s in fold_scores]}")

        # held-out test on FULL clouds with the best fold model
        print("\n===== Held-out test =====")
        os.makedirs("predictions", exist_ok=True)

        for fp in test_files:
            out = os.path.join("predictions",
                            os.path.basename(fp).replace(".las", "_pred.las"))
            segment_full_cloud(fp, best_model, CFG, out_path=out)
        torch.save(model,"PointTransformerV1.pth")

model loaded successfully
sample_data_0052.las | mIoU 0.191 (other 0.382, wood 0.000)
              precision    recall  f1-score   support

       other      1.000     0.382     0.553    501649
 wood_powder      0.000     0.000     0.000         0

    accuracy                          0.382    501649
   macro avg      0.500     0.191     0.276    501649
weighted avg      1.000     0.382     0.553    501649



/home/workstation-p/anaconda3/envs/dgcnn/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/workstation-p/anaconda3/envs/dgcnn/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/workstation-p/anaconda3/envs/dgcnn/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

## Next step — volume

`predictions/*_pred.las` files already carry the predicted `classification` field, so your existing 2.5D height-map volume code runs on them unchanged (filter `classification == 1`).

Remember the two volume-side fixes discussed earlier: floor plane from RANSAC as the height baseline (not `min(z)` of the pile), and interpolating empty grid cells instead of `nan_to_num -> 0`.

### Maximum error detect in segmentation
- 47
-